<a href="https://colab.research.google.com/github/PangingN/GAIA-DR3-Binary-Star-Detector/blob/main/154Proj.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Import Libararies & Access Drive**

In [ ]:
!pip install astropy
!pip install astroquery
!pip install lightkurve astroquery astropy matplotlib numpy
!pip install scipy
import pandas as pd
import lightkurve as lk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy as stats
from google.colab import drive
from astropy.timeseries import BoxLeastSquares
drive.mount('/content/drive')
from astroquery.nasa_exoplanet_archive import NasaExoplanetArchive
from astropy.coordinates import SkyCoord
import astropy.units as u
import seaborn as sns
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_predict, cross_val_score
from sklearn.metrics import accuracy_score, classification_report
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier

# **Access Lightcurve Data and Parameters of each Dataset from TESS**

### Query Exoplanets Data from TESS

In [ ]:
'''In this section I onvert RA to sexagesimal and Dec to sexagesimal format. This format
was eventually not needed, I tried to use it first to query the data until I realized TIC_IDs would
less tedious'''
planets = NasaExoplanetArchive.query_criteria(
    table="pscomppars",
    where="ra is not null and dec is not null") # choosing objects that do not have empty values for ra and dec

coords = SkyCoord(ra=planets['ra'], dec=planets['dec'], unit=(u.degree, u.degree))

planets['ra_sexagesimal'] = coords.ra.to_string(unit=u.hour, sep=':', precision=2)
planets['dec_sexagesimal'] = coords.dec.to_string(unit=u.degree, sep=':', precision=2)

# created a csv with about 6000 rows x 700 columns of exoplanets
planets.write("planets_sexagesimal.csv", format="csv", overwrite=True)

print("Conversion complete")

### Extract Exoplanet lightcurve data

In [ ]:
results = []

# Getting just the numeric part of the TIC ID e.g., "TIC 123456" → 123456
tic_ids = df['tic_id'].dropna().apply(lambda x: int(x.split(' ')[1]))

tic_ids = tic_ids[:100]

for i, tic in enumerate(tic_ids):

    print(f"{i+1}/{len(tic_ids)} : TIC {tic}")

    try: # searching for TESS SPOC light curves for this TIC ID

        search = lk.search_lightcurve(
            f"TIC {tic}", # Use f"TIC {tic}" for lightkurve search
            mission="TESS",
            author="SPOC")

        # skip if no light curves found
        if len(search) == 0:
            continue

        # stitch into one light curve
        lc = search.download_all().stitch()

        # remove NaNs, clip 5-sigma outliers, and normalize flux to median of 1
        lc = (
            lc.remove_nans()
              .remove_outliers(sigma=5)
              .normalize())

        # extract time, flux, and flux error arrays
        time = np.array(lc.time.value)
        flux = np.array(lc.flux.value)
        flux_err = np.array(lc.flux_err.value)

        # Mask any remaining non-finite values
        good = (
            np.isfinite(time)
            & np.isfinite(flux)
            & np.isfinite(flux_err))

        time = time[good]
        flux = flux[good]
        flux_err = flux_err[good]

        periods = np.linspace(0.5, 20, 10000)
        durations = np.linspace(0.02, 0.25, 10)

        # definining BLS search grid with 10000 trial periods and 10 trial durations
        bls = BoxLeastSquares(time, flux, dy=flux_err)
        power = bls.power(periods, durations)

        best_idx = np.argmax(power.power)

        results.append({
            "tic_id": tic,
            "period_days": float(power.period[best_idx]),
            "duration_days": float(power.duration[best_idx]),
            "transit_depth": float(power.depth[best_idx]),})

    except Exception as e:
        # log any failures and continue to next object
        print(f"Failed TIC {tic}: {e}")

results_df = pd.DataFrame(results)

results_df.to_csv('/content/drive/MyDrive/154/bls_results.csv', index=False)

results_df

### Access eclipsing binaries TIC_ID data from TESS villanova

In [ ]:
''' because at this point I found it was difficult to access query PM and RA's
for eclipsing binaries, I just accessed straight through the URL '''

url = "https://tessebs.villanova.edu/"


# get all HTML tables from the catalog webpage
tables = pd.read_html(url)

# access first table which contains the eclipsing binary catalog
eb_df = tables[0]
eb_df.head()

# get TIC IDs
eb_tic_ids = (
    eb_df["TESS ID"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .dropna()
    .astype(int))

# limit to the first 100 TIC IDs
eb_tic_ids.head()
eb_tic_ids_test = eb_tic_ids[:100]
print(eb_tic_ids_test)


NameError: name 'pd' is not defined

### Extract eclipsing binary lightcurve data

In [ ]:
import lightkurve as lk
import numpy as np
import pandas as pd
from astropy.timeseries import BoxLeastSquares

results = []

# Getting just the numeric part of the TIC ID e.g., "TIC 123456" → 123456
tic_ids = df['tic_id'].dropna().apply(lambda x: int(x.split(' ')[1]))

tic_ids = eb_tic_ids_test

for i, tic in enumerate(tic_ids):

    print(f"{i+1}/{len(tic_ids)} : TIC {tic}")

    try: # searching for TESS SPOC light curves for this TIC ID

        search = lk.search_lightcurve(
            f"TIC {tic}", # Use f"TIC {tic}" for lightkurve search
            mission="TESS",
            author="SPOC")

       # skip if no light curves found
        if len(search) == 0:
            continue

        # stitch into one light curve
        lc = search.download_all().stitch()

        # remove NaNs, clip 5-sigma outliers, and normalize flux to median of 1
        lc = (
            lc.remove_nans()
              .remove_outliers(sigma=5)
              .normalize())


        # extract time, flux, and flux error arrays
        time = np.array(lc.time.value)
        flux = np.array(lc.flux.value)
        flux_err = np.array(lc.flux_err.value)

        # Mask any remaining non-finite values
        good = (
            np.isfinite(time)
            & np.isfinite(flux)
            & np.isfinite(flux_err))

        time = time[good]
        flux = flux[good]
        flux_err = flux_err[good]

        periods = np.linspace(0.5, 20, 10000)
        durations = np.linspace(0.02, 0.25, 10)

        # definining BLS search grid with 10000 trial periods and 10 trial durations
        bls = BoxLeastSquares(time, flux, dy=flux_err)
        power = bls.power(periods, durations)

        best_idx = np.argmax(power.power)

        results.append({
            "tic_id": tic,
            "period_days": float(power.period[best_idx]),
            "duration_days": float(power.duration[best_idx]),
            "transit_depth": float(power.depth[best_idx]),})

    except Exception as e:

        # log any failures and continue to next object
        print(f"Failed TIC {tic}: {e}")

results_df = pd.DataFrame(results)

results_df.to_csv('/content/drive/MyDrive/154/eclipsingparameters.csv', index=False)

results_df

# **Perform KS Test on the three features**

### Feature 1: Period

In [ ]:
# access exoplanets through saved csv
exoplanets = '/content/drive/MyDrive/154/planetparameters.csv'
exoplanets = pd.read_csv(exoplanets)

binaries = '/content/drive/MyDrive/154/eclipsingparameters.csv'
binaries = pd.read_csv(binaries)

# extract transit periods
exo_periods = exoplanets["period_days"]
eb_periods = binaries["period_days"]

# perform ks test
stats.kstest(exo_periods, eb_periods)

### Feature 2: Transit Depth

In [ ]:
# extract transit depth
exo_depths = exoplanets["transit_depth"]
eb_depths = binaries["transit_depth"]

# perform ks test
stats.kstest(exo_depths, eb_depths)

###Feature 3: Duration

In [ ]:
# extract transit depth
exo_duration = exoplanets["duration_days"]
eb_duration = binaries["duration_days"]

# perform ks test
stats.kstest(exo_duration, eb_duration)


### Graph D for each parameter

In [ ]:
# Sort data
exo_sorted = np.sort(exo_periods)
eb_sorted = np.sort(eb_periods)

# ECDF values
exo_ecdf = np.arange(1, len(exo_sorted)+1) / len(exo_sorted)
eb_ecdf = np.arange(1, len(eb_sorted)+1) / len(eb_sorted)

plt.figure(figsize=(8,5))

plt.step(exo_sorted, exo_ecdf,
         where='post',
         label='Exoplanets')

plt.step(eb_sorted, eb_ecdf,
         where='post',
         label='Eclipsing Binaries')

plt.axvline(6.163, color='red',
            linestyle='--',
            label='Max KS Difference')

plt.xlabel('Period (days)')
plt.ylabel('Cumulative Fraction')
plt.title('KS Test Comparison of Orbital Periods')
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
exo_sorted = np.sort(exo_depths)
eb_sorted = np.sort(eb_depths)

exo_ecdf = np.arange(1, len(exo_sorted)+1) / len(exo_sorted)
eb_ecdf = np.arange(1, len(eb_sorted)+1) / len(eb_sorted)

plt.figure(figsize=(8,5))

plt.step(exo_sorted, exo_ecdf, where='post', label='Exoplanets')
plt.step(eb_sorted, eb_ecdf, where='post', label='Eclipsing Binaries')

plt.axvline(0.007, color='red', linestyle='--', label='Max KS Difference')

plt.xlabel('Transit Depth (Fractional Flux Drop)')
plt.ylabel('Cumulative Fraction')
plt.title('KS Test Comparison of Transit Depth')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
exo_sorted = np.sort(exo_duration)
eb_sorted = np.sort(eb_duration)

exo_ecdf = np.arange(1, len(exo_sorted)+1) / len(exo_sorted)
eb_ecdf = np.arange(1, len(eb_sorted)+1) / len(eb_sorted)

plt.figure(figsize=(8,5))

plt.step(exo_sorted, exo_ecdf, where='post', label='Exoplanets')
plt.step(eb_sorted, eb_ecdf, where='post', label='Eclipsing Binaries')

plt.axvline(0.224, color='red', linestyle='--', label='Max KS Difference')

plt.xlabel('Transit Duration (Days)')
plt.ylabel('Cumulative Fraction')
plt.title('KS Test Comparison of Transit Duration')
plt.legend()
plt.grid(True)
plt.show()

# **Perform Classification on the three features**

### Access and read data from drive

In [ ]:
# reading queried data containing the three parameters from both eb and exoplanets obtained from previous KS tests
exoplanet_and_binary = pd.read_csv('/content/sample_data/parameters for both for clustering - Copy of planetparameters.csv')
exoplanet_and_binary = exoplanet_and_binary.drop(columns='tic_id')
exoplanet_and_binary

### Create pairplots

In [ ]:
# creating a pairplot with raw data to see if there is any natural clustering
sns.pairplot(exoplanet_and_binary, hue='label')
plt.show()

### Prepare for classifiers

In [ ]:
# Fix spilt data into 70% training and 30% testing
# Using random_state = 11 so results are consistant and ensure reproducibility
X = exoplanet_and_binary.drop(columns='label')
y = exoplanet_and_binary['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=11)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"\nClass distribution in training set:\n{y_train.value_counts()}")
print(f"\nClass distribution in test set:\n{y_test.value_counts()}")

NameError: name 'exoplanet_and_binary' is not defined

In [ ]:
# function to calculating accuracy
def cv_accuracy(pipeline, X, y, label=""):
    scores = cross_val_score(pipeline, X, y,
                             scoring='accuracy')
    print(f"{label:40s}  CV accuracy: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores

### KNN

In [ ]:
# K-nearest neighbor algorithm
# k=3
pipe3 = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=3))])  # pipeline to chain data together

# CV accuracy using cv_accuracy function
cv_accuracy(pipe3, X_train, y_train, label="KNN k=3")

# scaling X_train since the pipe doesn't actually modify it outside of the function
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train.columns)

predict_3_cv = cross_val_predict(pipe3, X_train, y_train)
plotting_df_3cv = X_train_scaled_df.copy()
plotting_df_3cv['predicted_label_3cv'] = predict_3_cv

# plotting pairplot for k=3
sns.pairplot(plotting_df_3cv, hue='predicted_label_3cv')
plt.show()

In [ ]:
# k=5
pipe5 = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=5))])

# CV accuracy
cv_accuracy(pipe5, X_train, y_train, label="KNN k=5")

predict_5_cv = cross_val_predict(pipe5, X_train, y_train)
plotting_df_5cv = X_train_scaled_df.copy()
plotting_df_5cv['predicted_label_5cv'] = predict_5_cv

# plotting pairplot for k=5
sns.pairplot(plotting_df_5cv, hue='predicted_label_5cv')
plt.suptitle("KNN k = 5 — cross-val predictions (train)", y=1)
plt.show()

In [ ]:
# k=50
pipe50 = Pipeline([('scaler', StandardScaler()), ('knn', KNeighborsClassifier(n_neighbors=50))])

# CV accuracy
cv_accuracy(pipe50, X_train, y_train, label="KNN k=50")

predict_50_cv = cross_val_predict(pipe50, X_train, y_train)
plotting_df_50cv = X_train_scaled_df.copy()
plotting_df_50cv['predicted_label_50cv'] = predict_50_cv

# plotting pairplot for k=50
sns.pairplot(plotting_df_50cv, hue='predicted_label_50cv')
plt.suptitle("KNN k = 50 — cross-val predictions (train)", y=1)
plt.show()

In [ ]:
# classification reports for three k values
print("KNN k=3 Classification Report (Cross-Val on Train Set)")
print(classification_report(y_train, predict_3_cv, target_names=exoplanet_and_binary['label'].unique()))

print("KNN k=5 Classification Report (Cross-Val on Train Set)")
print(classification_report(y_train, predict_5_cv, target_names=exoplanet_and_binary['label'].unique()))

print("KNN k=50 Classification Report (Cross-Val on Train Set)")
print(classification_report(y_train, predict_50_cv, target_names=exoplanet_and_binary['label'].unique()))

In [ ]:
# plotting k values vs CV accuracy to check for where the accuracy drops
k_values = range(1, 71)
cv_means = []
cv_stds  = []

for k in k_values:
    pipe = Pipeline([('scaler', StandardScaler()),
                     ('knn', KNeighborsClassifier(n_neighbors=k))])
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds  = np.array(cv_stds)

plt.figure(figsize=(10, 5))
plt.plot(k_values, cv_means, marker='o', markersize=3, label='CV accuracy')
plt.fill_between(k_values, cv_means - cv_stds, cv_means + cv_stds, alpha=0.2, label='±1 std')
plt.xlabel('k')
plt.ylabel('CV Accuracy')
plt.title('KNN Accuracy vs. k')
plt.legend()
plt.show()

In [ ]:
# creating confusion matrix using k=5
pipe5.fit(X_train, y_train)
y_pred = pipe5.predict(X_test)

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                               display_labels=pipe5.classes_)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix - KNN k=5 (test)")
plt.show()

### Random Forest

In [ ]:
# random forest algorithm
rf_pipe = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(n_estimators=100, random_state=11))])

# CV accuracy
cv_accuracy(rf_pipe, X_train, y_train, label="Random Forest")

rf_predict_cv = cross_val_predict(rf_pipe, X_train, y_train)
plot_df_rf = X_train_scaled_df.copy()
plot_df_rf['predicted_label_rf'] = rf_predict_cv

# plotting rf pairplot
sns.pairplot(plot_df_rf, hue='predicted_label_rf')
plt.suptitle("Random Forest — cross-val predictions (train)", y=1)
plt.show()

In [ ]:
# classification report from rf
print("\nRandom Forest Classification Report (CV on train)")
rf_pipe.fit(X_train, y_train)
print(classification_report(y_train, rf_predict_cv, target_names=rf_pipe.classes_))

### Feature importance

In [ ]:
# feature importance of the three parameters
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

importances = pd.Series(rf.feature_importances_, index=X.columns)
importances.sort_values().plot(kind='barh')
plt.title("Feature Importances (Random Forest)")
plt.show()

### Confusion matrix

In [ ]:
# confusion matrix from rf
cm_rf = confusion_matrix(y_test, y_pred_rf)
disp_rf = ConfusionMatrixDisplay(confusion_matrix=cm_rf,
                               display_labels=rf_pipe.classes_)
disp_rf.plot(cmap='Greens')
plt.title("Confusion Matrix - Random Forest (test)")
plt.show()